In [ ]:

import pandas as pd
import numpy as np
from packaging.version import Version
from sklearn import __version__ as skl_version
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import deeproot as dr


# --- (A) Helper: OneHotEncoder compatível ---
def make_ohe():
    if Version(skl_version) >= Version("1.2"):
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    else:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# Carrega dataset limpo
df_clean = dr.load_data("telco_clean")
    
# Define X, y
y = df_clean["Churn"].astype(int)
X = df_clean.drop(columns=["Churn"])

# 3) Separar colunas categóricas e numéricas
cat_cols = [c for c in X.columns if (X[c].dtype == "object" or str(X[c].dtype) == "string")]
num_cols = [c for c in X.columns if c not in cat_cols]

# 4) Pré-processamento
cat_pipe = make_pipeline(make_ohe())
num_pipe = make_pipeline(StandardScaler())

pre = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, cat_cols),
        ("num", num_pipe, num_cols),
    ],
    remainder="drop",
)

# 4.1) Classificador base
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
)

# 4.2) Montar pipeline
pipe = Pipeline(steps=[
    ("prep", pre),
    ("rf", rf),
])

# 5) Split estratificado holdout para avaliação final
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)

# Predições e probabilidades
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

# Métricas
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test, y_pred, zero_division=0
)
auc = roc_auc_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)

print(f"✅ Treinado! ACC={acc:.4f} | F1macro={f1_macro:.4f} | AUC={auc:.4f}")
print("Matriz de confusão:\n", cm)

# Salva métricas
metrics_payload = {
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "auc": float(auc),
    "per_class": [
        {"label": str(lbl), "precision": float(p), "recall": float(r), "f1": float(f1v), "support": int(s)}
        for lbl, p, r, f1v, s in zip(sorted(y.unique()), precision, recall, f1_per_class, support)
    ],
    "confusion_matrix": cm.tolist(),
    "classification_report": report,
    "n_features": {"categorical": len(cat_cols), "numerical": len(num_cols)},
}

# Salva modelo
dr.save_model(pipe, "telcoChurn", train_data="telco_clean", metrics=metrics_payload)
print("✅ Modelo e métricas salvos como telcoChurn")
